# Bitcoin OTC — preprocessing and verification

The published artifact `Bitcoin-OTC-trust_threshold-with-both-community-labels-scaled.csv`
is **not reproducible edge for edge**: the original augmentation drew random targets with
no seed, and the two community labelings came from external Gephi and networkx runs.

This notebook therefore does two separate jobs:

* **Rebuild** — run the documented pipeline from the source data with an explicit seed,
  producing a dataset with the same construction and summary statistics but a different
  random draw. Use this to study sensitivity, not to regenerate the artifact.
* **Verify** — load the published artifact and check every invariant that must hold
  regardless of the random draw: observed edges identical to the source, scaling formula,
  out-degree floor, node-consistent labels, and the sink structure under the gate.

```
Bitcoin-OTC-trust.csv                    35,592 edges / 5,881 nodes
        |
        |  Step 1  out-degree augmentation: +41,022 neutral edges (Sign = 0)
        |  Step 2  largest weakly connected component (no-op here)
        |  Step 3  min-max scaling of Sign into [0, 1]
        |  Step 4  join two community labelings onto both endpoints
        v
...-with-both-community-labels-scaled.csv  76,614 edges
```

Requires `pandas`, `numpy`, `networkx`.

## Configuration

In [ ]:
from pathlib import Path
from typing import Dict, Tuple

import networkx as nx
import numpy as np
import pandas as pd

DATA_DIR = Path(".")

RAW_CSV       = DATA_DIR / "Bitcoin-OTC-trust.csv"
LABELS_CSV    = DATA_DIR / "node_community_labels.csv"
ARTIFACT_CSV  = DATA_DIR / "Bitcoin-OTC-trust_threshold-with-both-community-labels-scaled.csv"
REBUILT_CSV   = DATA_DIR / "Bitcoin-OTC-rebuilt.csv"

OUT_DEGREE_FLOOR = 10      # threshold in augment_network
SEED = 42                  # the original run was unseeded; see the header

# Trust/distrust gate: an edge survives if T >= T_T and U <= U_T, where
#   T = trust - distrust        U = trust + distrust - 1
# Added links carry the neutral pair (trust, distrust) = (0.5, 0.5) -> T = 0, U = 0.
GATES = {"loose": (-0.8, 0.0), "strict": (0.2, 0.8)}

FINAL_COLS = ["source", "target", "Sign",
              "source_com_label", "target_com_label", "Sign_scaled",
              "source_com_label_new", "target_com_label_new"]

## Helpers

In [ ]:
def load_raw(path: Path = RAW_CSV) -> pd.DataFrame:
    """Source ratings, renamed to the column convention of the artifact."""
    df = pd.read_csv(path).rename(columns={"From": "source", "To": "target"})
    assert (df["Sign"] != 0).all(), "Sign = 0 must not occur in the source data"
    assert not df.duplicated(["source", "target"]).any()
    assert (df["source"] != df["target"]).all()
    return df[["source", "target", "Sign"]]


def build_digraph(df: pd.DataFrame) -> nx.DiGraph:
    g = nx.DiGraph()
    g.add_nodes_from(pd.concat([df["source"], df["target"]]).unique().tolist())
    g.add_edges_from(df[["source", "target"]].astype(int).itertuples(index=False, name=None))
    return g


def describe(df: pd.DataFrame, label: str) -> dict:
    """Structure summary, including the sink count that motivates Step 1."""
    g = build_digraph(df)
    if g.number_of_edges() == 0:
        print(f"[{label}] empty graph")
        return {"label": label, "edges": 0}
    sccs = list(nx.strongly_connected_components(g))
    cond = nx.condensation(g, sccs)
    info = {"label": label,
            "nodes": g.number_of_nodes(),
            "edges": g.number_of_edges(),
            "wcc": nx.number_weakly_connected_components(g),
            "sccs": len(sccs),
            "largest_scc": max(len(s) for s in sccs),
            "sink_sccs": sum(1 for n, d in cond.out_degree() if d == 0)}
    print(f"[{label:26s}] nodes={info['nodes']:5d} edges={info['edges']:6d} "
          f"WCC={info['wcc']:4d} SCC={info['sccs']:5d} "
          f"largestSCC={info['largest_scc']:5d} sinkSCC={info['sink_sccs']:5d}")
    return info

## Step 1 — Out-degree augmentation

For each node with out-degree below the floor, neutral edges (`Sign = 0`) are drawn to
uniformly random non-neighbours, excluding self-loops and duplicates, until the node
reaches the floor. Direction is always outgoing from the deficient node; in-degree is
never targeted, and nodes already at or above the floor are untouched.

The floor exists to keep the network usable once the gate is applied: in the source
network 1,067 of 5,881 nodes have out-degree 0 and 1,082 of 1,144 strongly connected
components are sinks. Neutral links clear the loose gate (`T = 0 >= -0.8`,
`U = 0 <= 0`) while asserting neither trust nor distrust.

In [ ]:
def augment_out_degree(df: pd.DataFrame, floor: int = OUT_DEGREE_FLOOR,
                       seed: int = SEED) -> pd.DataFrame:
    """Add neutral (Sign = 0) out-edges until every node reaches `floor`."""
    rng = np.random.default_rng(seed)
    nodes = set(df["source"]) | set(df["target"])

    out_neighbours: Dict[int, set] = {v: set() for v in nodes}
    for s, t in df[["source", "target"]].itertuples(index=False, name=None):
        out_neighbours[s].add(t)

    new_edges = []
    for u in sorted(nodes):                       # sorted -> deterministic given seed
        deficit = floor - len(out_neighbours[u])
        if deficit <= 0:
            continue
        candidates = np.array(sorted(nodes - out_neighbours[u] - {u}))
        k = min(deficit, len(candidates))
        for v in rng.choice(candidates, k, replace=False):
            new_edges.append({"source": u, "target": int(v), "Sign": 0})
            out_neighbours[u].add(int(v))

    return pd.concat([df, pd.DataFrame(new_edges)], ignore_index=True)


def keep_largest_wcc(df: pd.DataFrame) -> pd.DataFrame:
    """Step 2. Retained for fidelity to the original pipeline; a no-op after Step 1."""
    g = build_digraph(df)
    largest = max(nx.weakly_connected_components(g), key=len)
    keep = df["source"].isin(largest) & df["target"].isin(largest)
    dropped = (~keep).sum()
    print(f"largest WCC keeps {len(largest)} nodes; dropped {dropped} edges")
    return df.loc[keep].reset_index(drop=True)


def scale_sign(df: pd.DataFrame) -> pd.DataFrame:
    """Step 3. Min-max scaling of Sign into [0, 1]; equals (Sign + 10) / 20 here."""
    out = df.copy()
    lo, hi = out["Sign"].min(), out["Sign"].max()
    out["Sign_scaled"] = (out["Sign"] - lo) / (hi - lo)
    print(f"Sign range [{lo}, {hi}] -> Sign_scaled [0, 1]; neutral maps to "
          f"{out.loc[out['Sign'] == 0, 'Sign_scaled'].iloc[0]:.2f}")
    return out

In [ ]:
raw = load_raw()
describe(raw, "source data")

rebuilt = scale_sign(keep_largest_wcc(augment_out_degree(raw)))
describe(rebuilt, "rebuilt (seeded)")

## Step 4 — Community labels

The two labelings are external inputs, not computed here: one Gephi run (resolution 1.4,
modularity 0.467) and one networkx run on the 76,614-edge graph. `node_community_labels.csv`
holds them as a node table so they can be joined onto any edge list over the same nodes;
the `_new` column is the 2,566 / 3,315 split used in the simulations.

In [ ]:
def attach_labels(df: pd.DataFrame, labels_path: Path = LABELS_CSV) -> pd.DataFrame:
    """Join the node-level community labels onto both endpoints."""
    labels = pd.read_csv(labels_path)
    assert labels["node"].is_unique, "label table must be one row per node"

    out = df.copy()
    for end in ("source", "target"):
        renamed = labels.rename(columns={
            "node": end,
            "com_label": f"{end}_com_label",
            "com_label_new": f"{end}_com_label_new"})
        out = out.merge(renamed, on=end, how="left")

    missing = out[[c for c in out.columns if c.endswith("com_label")]].isna().any(axis=1).sum()
    assert missing == 0, f"{missing} edges reference nodes absent from the label table"
    return out[FINAL_COLS]


rebuilt = attach_labels(rebuilt)
rebuilt.to_csv(REBUILT_CSV, index=False)
print(f"wrote {REBUILT_CSV.name}: {len(rebuilt):,} edges")
rebuilt.head()

## Verification of the published artifact

Everything below must hold for the shipped file regardless of the random draw. This is
the cell to run after any change upstream.

In [ ]:
artifact = pd.read_csv(ARTIFACT_CSV)
observed = artifact[artifact["Sign"] != 0]
added = artifact[artifact["Sign"] == 0]

key = ["source", "target", "Sign"]
assert (artifact.columns.tolist() == FINAL_COLS), "unexpected column layout"
assert observed[key].sort_values(key).reset_index(drop=True).equals(
    raw[key].sort_values(key).reset_index(drop=True)), "observed edges differ from source"
assert not artifact.duplicated(["source", "target"]).any(), "duplicate edge"
assert (artifact["source"] != artifact["target"]).all(),    "self-loop"
assert artifact.notna().all().all(),                        "missing values"
assert np.allclose(artifact["Sign_scaled"], (artifact["Sign"] + 10) / 20), "scaling formula"
assert artifact.groupby("source").size().min() >= OUT_DEGREE_FLOOR, "out-degree floor"

for end in ("source", "target"):
    for col in (f"{end}_com_label", f"{end}_com_label_new"):
        assert artifact.groupby(end)[col].nunique().max() == 1, f"{col} not node-consistent"

print(f"all checks passed: {len(artifact):,} edges "
      f"({len(observed):,} observed, {len(added):,} neutral = "
      f"{len(added) / len(artifact):.0%})")
print(f"neutral edges shared with this seeded rebuild: "
      f"{len(set(map(tuple, added[['source','target']].to_numpy())) & set(map(tuple, rebuilt.loc[rebuilt.Sign == 0, ['source','target']].to_numpy())))}"
      f" of {len(added):,} - the original draw was unseeded")

## Sensitivity checks

Two questions a reviewer is likely to ask: does the gate leave the network usable, and
was a floor of 10 necessary?

In [ ]:
def gate_survivors(df: pd.DataFrame, gate: Tuple[float, float]) -> pd.DataFrame:
    """Apply the gate under (trust, distrust) = (Sign_scaled, 1 - Sign_scaled) as a
    stand-in: neutral edges then sit at T = 0, U = 0, matching the (0.5, 0.5) pair
    the original pipeline assigned them."""
    tt, ut = gate
    trust = df["Sign_scaled"]
    distrust = 1 - trust
    T, U = trust - distrust, trust + distrust - 1
    return df[(T >= tt) & (U <= ut)]


for name, gate in GATES.items():
    describe(gate_survivors(artifact, gate),          f"artifact / gate {name}")
    describe(gate_survivors(artifact[artifact.Sign != 0], gate), f"observed only / gate {name}")

In [ ]:
# Was a floor of 10 needed for connectivity?
rows = []
for floor in (1, 2, 3, 5, 10):
    aug = augment_out_degree(raw, floor=floor, seed=SEED)
    g = build_digraph(aug)
    rows.append({"floor": floor,
                 "added_edges": len(aug) - len(raw),
                 "SCCs": nx.number_strongly_connected_components(g),
                 "WCCs": nx.number_weakly_connected_components(g)})
pd.DataFrame(rows).set_index("floor")

## Notes

* **Treat the published artifact as fixed.** The rebuild is a comparable dataset, not a
  regeneration: the neutral draw differs, and so would any Leiden or Gephi partition run
  on it.
* **Report results against `Sign != 0` as well** wherever a finding could depend on the
  placeholders, which are 54% of the edges.
* **The neutral pair sits on the boundary of the loose gate** (`U = 0 <= 0`). A negative
  `U_T`, or a trust/distrust construction that pushes `U` above zero, discards all 41,022
  at once.
* **The loose gate still leaves sink components**, so a sink-resolution step of the kind
  used for the legislators dataset is needed before out-Laplacian dynamics can reach a
  single consensus.